# Forwards & backwards flows

This recipe demonstrates how forwards and backwards flows work.

For demonstration, the CSV data is written directly in the cell below -- in practice you would want to load data a file.

In [ ]:
import pandas as pd
from io import StringIO

flows = pd.read_csv(StringIO("""
source,target,type,value
a,b,main,2
a,c,main,1
c,d,main,3
b,c,back,2
"""))

flows

Here is one structure, with nodes `b` and `c` both in the same vertical slice:

In [ ]:
from floweaver import *

# Set the default size to fit the documentation better.
size = dict(width=570, height=300)

nodes = {
    'a': ProcessGroup(['a']),
    'b': ProcessGroup(['b']),
    'c': ProcessGroup(['c']),
    'd': ProcessGroup(['d']),
    'back': Waypoint(direction='L'),
}

bundles = [
    Bundle('a', 'b'),
    Bundle('a', 'c'),
    Bundle('b', 'c', waypoints=['back']),
    Bundle('c', 'd'),
    Bundle('c', 'b'),
]

ordering = [
    [['a'], []],
    [['b', 'c'], ['back']],
    [['d'], []],
]

sdd = SankeyDefinition(nodes, bundles, ordering)

weave(sdd, flows).to_widget(**size)

Alternatively, if `b` is moved to the right, extra hidden waypoints are automatically added to get the `b--c` flow back to the left of `c`:

In [ ]:
bundles = [
    Bundle('a', 'b'),
    Bundle('a', 'c'),
    Bundle('b', 'c'),
    Bundle('c', 'd'),
    Bundle('c', 'b'),
]

ordering = [
    [['a'], []],
    [['c'], ['back']],
    [['b', 'd'], []],
]

sdd = SankeyDefinition(nodes, bundles, ordering)

weave(sdd, flows).to_widget(**size)

## Loops to the same process

The best way to show this is with a waypoint located just below the process, to ensure there is space in the diagram for the loop.

In [ ]:
flows = pd.read_csv(StringIO("""
source,target,type,value
a,b,main,2
b,c,main,2
b,b,main,1
"""))

flows

In [ ]:
nodes = {
    'a': ProcessGroup(['a']),
    'b': ProcessGroup(['b']),
    'c': ProcessGroup(['c']),
    'w': Waypoint(direction='L', title=''),
}

bundles = [
    Bundle('a', 'b'),
    Bundle('b', 'c'),
    Bundle('b', 'b', waypoints=['w'], flow_selection='type == "main"'),
]

ordering = [
    [['a']],
    [['b'], ['w']],
    [['c']],
]

sdd = SankeyDefinition(nodes, bundles, ordering)

weave(sdd, flows).to_widget(**size)